# GWAD reproduction on Colab

CVPR 2025 *Mind the Gap: Detecting Black-box Adversarial Attacks in the Making through Query Update Analysis*.

1. `Runtime > Change runtime type > T4 GPU`
2. Run cells top to bottom. Everything persists in Google Drive (`MyDrive/GWAD`), so a disconnect or a new session loses nothing:
   - `repo/`: the code, cloned once. It is **not** updated automatically; set `UPDATE_MAIN = True` in the clone cell to pull the latest main. Weights and CIFAR-10 also live here (gitignored), downloaded once.
   - `runs/<time>_<attack>_<scenario>/`: log + DS histograms / distributions of each run, never overwritten.
   - `results/<time>_<run name>/`: summary images and GWAD timeline (`--save_dir`).
3. Colab free tier: max ~12h per session, so keep `N_IMAGES` small (HSJA ~1-2 min per image).

In [ ]:
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DRIVE_DIR = '/content/drive/MyDrive/GWAD'
REPO_DIR = f'{DRIVE_DIR}/repo'
!mkdir -p {DRIVE_DIR}/runs {DRIVE_DIR}/results {DRIVE_DIR}/notebooks

import json, os, time
from google.colab import _message

def save_notebook(flush=False):
    """Save this notebook, outputs included, to Drive.

    Goes to RUN_DIR/notebook.ipynb when a run was started in this session, otherwise to
    DRIVE_DIR/notebooks/<time>.ipynb. Existing files are never overwritten (a new run gets a new RUN_DIR).
    flush=True also forces Drive to finish uploading (unmounts Drive; rerun the mount cell to use it again).
    """
    nb = _message.blocking_request('get_ipynb', timeout_sec=120)['ipynb']
    run_dir = globals().get('RUN_DIR')
    if run_dir:
        path = f'{run_dir}/notebook.ipynb'
        if os.path.exists(path):
            path = f"{run_dir}/notebook_{time.strftime('%H%M%S')}.ipynb"
    else:
        path = f"{DRIVE_DIR}/notebooks/{time.strftime('%Y%m%d-%H%M%S')}.ipynb"
    tmp = path + '.tmp'
    with open(tmp, 'w', encoding='utf-8') as f:
        json.dump(nb, f, ensure_ascii=False)
        f.flush()
        os.fsync(f.fileno())
    os.replace(tmp, path)
    with open(path, encoding='utf-8') as f:  # read back to make sure the file is complete
        n_cells = len(json.load(f)['cells'])
    print('notebook saved: %s (%d cells, %.1f KB)' % (path, n_cells, os.path.getsize(path) / 1024))
    if flush:
        drive.flush_and_unmount()
        print('Drive flushed, upload finished')
    return path

In [ ]:
import os
REPO_URL = 'https://github.com/Chu-c-git/GWAD_reproduction.git'
UPDATE_MAIN = False  # True: pull the latest main (tracked stats/ files are reset first; every run is archived in runs/)

if not os.path.exists(f'{REPO_DIR}/.git'):
    !git clone -q {REPO_URL} {REPO_DIR}
elif UPDATE_MAIN:
    !git -C {REPO_DIR} checkout -q -- stats
    !git -C {REPO_DIR} pull -q --ff-only
%cd {REPO_DIR}
!git log -1 --format='code version: %h %s (%cd)' --date=short
# torch / torchvision / numpy / scikit-image / opencv / matplotlib / tqdm are preinstalled on Colab
!pip install -q gdown truststore

## Target model weights + CIFAR-10 (cached in Drive)

In [ ]:
# weights and CIFAR-10 are kept inside the Drive repo (gitignored), so they are downloaded once;
# reuse the cache of the old layout (DRIVE_DIR/model, DRIVE_DIR/data) if it exists
if not os.path.exists('model/cifar10/resnet18.pt'):
    if os.path.exists(f'{DRIVE_DIR}/model/resnet18.pt'):
        !mkdir -p model/cifar10 && cp {DRIVE_DIR}/model/resnet18.pt model/cifar10/
    else:
        !python scripts/download_weights.py
if not os.path.exists('data/cifar-10-batches-py') and os.path.exists(f'{DRIVE_DIR}/data/cifar-10-batches-py'):
    !mkdir -p data && cp -r {DRIVE_DIR}/data/cifar-10-batches-py data/
!python -c "from torchvision import datasets; datasets.CIFAR10('./data', train=False, download=True)"
!ls -la model/cifar10 data/

## Run

- `ATTACK`: `hsja`, `nes`, `sign-flip`
- `SCENARIO`: `benign`, `standard`, `batch`
- `N_IMAGES`: benign = number of benign queries, attack = number of attacked images (batch scenario first uses 2000 images as benign pool, these are not counted)

In [ ]:
import time
ATTACK = 'hsja'
SCENARIO = 'standard'
N_IMAGES = 5

# every run gets its own folder in Drive: runs/<time>_<attack>_<scenario>/
RUN_DIR = f"{DRIVE_DIR}/runs/{time.strftime('%Y%m%d-%H%M%S')}_{ATTACK}_{SCENARIO}"
!mkdir -p {RUN_DIR} && touch {RUN_DIR}/.start
!python demo.py --data cifar10 --attack {ATTACK} --scenario {SCENARIO} --n_images {N_IMAGES} --save_dir {DRIVE_DIR}/results 2>&1 | tee {RUN_DIR}/log.txt

In [ ]:
# archive the DS histograms / DS distributions this run wrote to stats/ (the next run reuses the same file names)
!cd stats && find hist dist -type f -newer {RUN_DIR}/.start -exec cp --parents {} {RUN_DIR}/ \;
!ls -R {RUN_DIR}

## Plot mean delta-similarity histogram (2001 bins over [-1, 1], not the 201-bin HoDS; Python port of `stats/show_stats.m`)

In [ ]:
import glob
import numpy as np
import matplotlib.pyplot as plt

plt.figure(figsize=(8, 4))
for f in sorted(glob.glob('stats/hist/*.txt')):
    h = np.loadtxt(f, delimiter=',')
    plt.plot(np.linspace(-1, 1, len(h)), h, label=os.path.basename(f)[:-4])
plt.xlabel('delta similarity (cosine)')
plt.ylabel('mean count')
plt.legend()
if 'RUN_DIR' in globals():
    plt.savefig(f'{RUN_DIR}/hist.png', dpi=150, bbox_inches='tight')
plt.show()

## Save this notebook (with outputs) to Drive

Run this cell last. It writes the notebook with all outputs to `runs/<this run>/notebook.ipynb` (or `notebooks/<time>.ipynb` when no run was started), then flushes Drive so the upload has finished before you close the tab.

In [ ]:
save_notebook(flush=True)